# Línea base y respuestas de Qwen3-1.7B en Kaggle (IIC-3694)

Versión para Kaggle de `qwen_objetivo_colab.ipynb`. Hace lo mismo: mide la línea base sobre `eval` (pass@1 y pass@5) y guarda respuestas sobre `train` para armar los datos SFT/DPO. **Continúa una corrida anterior** si se la das como entrada: solo corre las muestras que faltan.

Pasos (detalle en `docs/guia_kaggle.md`):

1. Crea un *dataset* de Kaggle con la carpeta `inventory_harness` de tu Drive (trae el zip del código y `results/`).
2. En este cuaderno: *Add Input* con ese dataset; en *Settings*, **Accelerator: GPU T4 x2** e **Internet: On**.
3. *Save Version* → **Save & Run All (Commit)**. Corre en segundo plano, con el computador apagado, hasta 12 horas.
4. Al terminar, descarga `resultados_qwen17b.zip` desde la pestaña *Output* de esa versión.

Con dos GPU, la línea base corre en una y las respuestas de `train` en la otra, al mismo tiempo.

## 1. GPU

In [ ]:
import os
import time

INICIO = time.time()
PRUEBA_SECA = os.environ.get("PRUEBA_SECA") == "1"  # solo para probar el cuaderno sin GPU

if PRUEBA_SECA:
    N_GPU = int(os.environ.get("N_GPU", "2"))
else:
    import torch

    N_GPU = torch.cuda.device_count()
    for i in range(N_GPU):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
if N_GPU == 0:
    print("=" * 70)
    print("ATENCION: NO HAY GPU. Settings > Accelerator > GPU T4 x2, y vuelve a ejecutar.")
    print("=" * 70)

## 2. Encontrar el código y los resultados anteriores

Busca en `/kaggle/input` el zip del código (o la carpeta, si Kaggle lo descomprimió) y las carpetas `linea_base_qwen17b` y `trazas_qwen17b` de una corrida anterior. Las copia a `/kaggle/working`, que es lo que Kaggle guarda al final.

In [ ]:
import shutil
import sys
import zipfile
from pathlib import Path

ENTRADA = Path(os.environ.get("ENTRADA_KAGGLE", "/kaggle/input"))
SALIDA = Path(os.environ.get("SALIDA_KAGGLE", "/kaggle/working"))
CODIGO = Path(os.environ.get("CODIGO_KAGGLE", "/tmp/inventory_harness"))
RESULTADOS = SALIDA / "results"
RESULTADOS.mkdir(parents=True, exist_ok=True)
CARPETAS = ("linea_base_qwen17b", "trazas_qwen17b")


def buscar_codigo():
    for ruta in sorted(ENTRADA.glob("**/inventory_harness_colab.zip")):
        if ruta.is_file():
            return ruta
    for script in sorted(ENTRADA.glob("**/scripts/run_pilot.py")):
        raiz = script.parent.parent
        if (raiz / "src" / "inventory_harness").is_dir():
            return raiz
    return None


origen = buscar_codigo()
if origen is None:
    raise SystemExit("No encontre el codigo en /kaggle/input. Agrega el dataset con Add Input.")
shutil.rmtree(CODIGO, ignore_errors=True)
if origen.is_file():
    with zipfile.ZipFile(origen) as archivo:
        archivo.extractall(CODIGO)
else:
    shutil.copytree(origen, CODIGO)
sys.path.insert(0, str(CODIGO / "src"))
print("Codigo desde", origen)


def rollouts_hechos(carpeta):
    total = 0
    for archivo in carpeta.glob("rollouts_*.jsonl"):
        with archivo.open(encoding="utf-8", errors="replace") as f:
            total += sum(1 for linea in f if linea.strip())
    return total


for nombre in CARPETAS:
    destino = RESULTADOS / nombre
    previas = sorted(m.parent for m in ENTRADA.glob(f"**/{nombre}/manifest.json"))
    if destino.exists():
        print(f"{nombre}: ya esta en {destino} ({rollouts_hechos(destino)} rollouts); se continua.")
    elif previas:
        shutil.copytree(previas[0], destino)
        print(f"{nombre}: copiada desde {previas[0]} ({rollouts_hechos(destino)} rollouts); se continua.")
        if len(previas) > 1:
            print("  AVISO: habia varias; se uso la primera:", *previas, sep="\n  ")
    else:
        print(f"{nombre}: no hay corrida anterior; empieza de cero.")

## 3. Dependencias y pruebas del núcleo

In [ ]:
import subprocess

if not PRUEBA_SECA:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51", "accelerate", "pytest"],
        check=True,
    )

In [ ]:
prueba = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_solver.py",
     "tests/test_numbers_es.py", "tests/test_verifier.py", "tests/test_generator.py"],
    cwd=CODIGO, capture_output=True, text=True,
)
print(prueba.stdout[-500:], prueba.stderr[-500:])

## 4. Configuración

`LINEA_BASE_*` **no se cambia**: debe ser igual a la corrida anterior para poder continuarla. `TRAZAS_*` sí se puede bajar. `LIMITE_HORAS` detiene todo a tiempo para guardar antes del tope de 12 horas de Kaggle; lo que alcance a correr queda guardado y se puede continuar en otra versión.

In [ ]:
MODELO = "Qwen/Qwen3-1.7B"
BATCH_SIZE = 8  # con 16 no cabe en una T4 con 4096 tokens

LINEA_BASE_POR_CELDA = 10  # no cambiar
LINEA_BASE_MUESTRAS = 5  # no cambiar
TRAZAS_POR_CELDA = 10
TRAZAS_MUESTRAS = 2

LIMITE_HORAS = 11.0

# ------------------- No hace falta editar desde aquí -------------------
import threading

DIR_LINEA_BASE = RESULTADOS / "linea_base_qwen17b"
DIR_TRAZAS = RESULTADOS / "trazas_qwen17b"
DIR_DATOS = RESULTADOS / "datos_derivados_qwen17b"
LIMITE = INICIO + LIMITE_HORAS * 3600
SECO = ["--dry-run"] if PRUEBA_SECA else []

LINEA_BASE = [
    "scripts/run_pilot.py", "--stage", "piloto", "--models", MODELO, "--thinking", "on",
    "--n-per-cell", LINEA_BASE_POR_CELDA, "--n-samples", LINEA_BASE_MUESTRAS,
    "--batch-size", BATCH_SIZE, "--out", DIR_LINEA_BASE, *SECO,
]
TRAZAS = [
    "scripts/run_pilot.py", "--stage", "trazas", "--models", MODELO, "--thinking", "on",
    "--n-per-cell", TRAZAS_POR_CELDA, "--n-samples", TRAZAS_MUESTRAS,
    "--batch-size", BATCH_SIZE, "--out", DIR_TRAZAS, *SECO,
]


def lanzar(nombre, argumentos, gpu=None):
    """Inicia un script del proyecto y muestra su salida, con el nombre de la tanda."""
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    proceso = subprocess.Popen(
        [sys.executable, "-u", *map(str, argumentos)], cwd=CODIGO, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )

    def leer():
        for linea in proceso.stdout:
            print(f"[{nombre}] {linea}", end="", flush=True)

    hilo = threading.Thread(target=leer, daemon=True)
    hilo.start()
    return nombre, proceso, hilo


def esperar(*tandas):
    """Espera a las tandas; si se pasa LIMITE_HORAS, las detiene (lo hecho queda guardado)."""
    while any(proceso.poll() is None for _, proceso, _ in tandas):
        if time.time() > LIMITE:
            print(f"Se alcanzo LIMITE_HORAS={LIMITE_HORAS}: se detiene para guardar.", flush=True)
            for _, proceso, _ in tandas:
                if proceso.poll() is None:
                    proceso.terminate()
            break
        time.sleep(5 if PRUEBA_SECA else 30)
    for nombre, proceso, hilo in tandas:
        try:
            proceso.wait(timeout=120)
        except subprocess.TimeoutExpired:
            proceso.kill()
            proceso.wait()
        hilo.join(timeout=10)
        print(f"[{nombre}] termino con codigo {proceso.returncode}", flush=True)


print(f"Se detiene a las {LIMITE_HORAS} h de iniciado el cuaderno.")

## 5. Correr

Primero descarga el modelo una vez, para que las dos tandas no lo bajen a la vez. Con dos GPU corren en paralelo; con una, la línea base va primero. Cada ~25 respuestas se imprime el avance, y cada ~130 una línea `TOTAL` con el tiempo que falta.

In [ ]:
if not PRUEBA_SECA:
    from huggingface_hub import snapshot_download

    snapshot_download(MODELO)

if N_GPU >= 2:
    print("Dos GPU: linea base en la 0 y trazas en la 1, en paralelo.")
    esperar(lanzar("linea base", LINEA_BASE, gpu=0), lanzar("trazas", TRAZAS, gpu=1))
else:
    print("Una GPU: primero la linea base, despues las trazas.")
    esperar(lanzar("linea base", LINEA_BASE))
    if time.time() < LIMITE - 1800:
        esperar(lanzar("trazas", TRAZAS))
    else:
        print("No queda tiempo para las trazas en esta version.")
print(f"Tiempo total: {(time.time() - INICIO) / 3600:.2f} h")

## 6. Informe, datos SFT/DPO y zip

Corre aunque alguna tanda haya quedado incompleta: el informe y los datos se arman con lo que haya.

In [ ]:
from IPython.display import Markdown, display


def correr(*argumentos):
    resultado = subprocess.run(
        [sys.executable, "-u", *map(str, argumentos)], cwd=CODIGO, capture_output=True, text=True
    )
    print(resultado.stdout[-3000:], resultado.stderr[-3000:])
    return resultado.returncode


if (DIR_LINEA_BASE / "manifest.json").exists():
    print("Linea base:", rollouts_hechos(DIR_LINEA_BASE), "rollouts")
    correr("scripts/analyze_results.py", DIR_LINEA_BASE)
if (DIR_TRAZAS / "manifest.json").exists():
    print("Trazas:", rollouts_hechos(DIR_TRAZAS), "rollouts")
    correr("scripts/analyze_results.py", DIR_TRAZAS)
    correr("scripts/build_training_data.py", DIR_TRAZAS, "--out", DIR_DATOS)

reporte = DIR_LINEA_BASE / "report.md"
if reporte.exists():
    display(Markdown(reporte.read_text(encoding="utf-8")))

In [ ]:
import datetime

ZIP_RESULTADOS = SALIDA / "resultados_qwen17b.zip"
carpetas = [d for d in (DIR_LINEA_BASE, DIR_TRAZAS, DIR_DATOS) if d.exists()]
with zipfile.ZipFile(ZIP_RESULTADOS, "w", zipfile.ZIP_DEFLATED) as archivo:
    for carpeta in carpetas:
        for ruta in sorted(carpeta.iterdir()):
            if ruta.is_file() and not ruta.name.endswith(".tmp"):
                archivo.write(ruta, f"{carpeta.name}/{ruta.name}")
print(datetime.datetime.now().isoformat(timespec="seconds"))
print(f"LISTO ({ZIP_RESULTADOS.stat().st_size / 2**20:.1f} MiB): {ZIP_RESULTADOS}")
print("Descargalo desde la pestana Output de esta version.")